# 🎙️ 声音克隆：Qwen3-TTS 1.7B（Colab 免费 T4）

配合「声音克隆工作台」使用：先在工作台录好声音、输好文字，下载 **voice-pack.zip**，再按下面顺序运行。

**开始前：运行时 → 更改运行时类型 → 硬件加速器选 T4 GPU。**

流程：① 装依赖 → ② 上传语音包 → ③ 加载模型（首次约 1–2 分钟，3.86GB）→ ④ 逐段生成 → ⑤ 打包下载。

想直接走工作台 T4：跑完 ① 后直接跑 **⑥ 一键启动**（语音包已在 Google Drive 里，不用手动传），Gradio 公开链接会自动登记，工作台打开就自动用这个 T4 地址，不用再手动贴链接。

In [ ]:
# ① 检查 GPU + 安装依赖
import torch
print('CUDA 可用:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('显卡:', torch.cuda.get_device_name(0))
else:
    print('⚠️ 没连上 GPU！先点 运行时 → 更改运行时类型 → 选 T4 GPU，再重跑。')

# 安装 Qwen3-TTS 官方包（Colab 自带 torch，不用重装）
# 版本 pin 说明：qwen-tts 钉死 transformers==4.57.3，而它只允许 huggingface-hub<1.0；
# Colab 预装的 diffusers 0.42.1 却要求 huggingface-hub>=1.32.0，两边没有能同时满足的 hub 版本。
# diffusers 本就不是 qwen-tts 的依赖（只是 Colab 预装），所以反向把它降到 0.39.0——
# 这是最后一个兼容 hub<1.0 的版本（要求 hub<2.0,>=0.34.0），hub 保持 Colab 预装的 0.36.2。
# 升 transformers 到 5.x 行不通：qwen-tts 钉死 4.57.3，5.x 是大版本 breaking change。
!pip install -U -q qwen-tts soundfile "huggingface-hub==0.36.2" "transformers==4.57.3" "diffusers==0.39.0"

# 验证关键依赖版本
import huggingface_hub, diffusers, transformers
print('huggingface-hub:', huggingface_hub.__version__)
print('transformers:', transformers.__version__)
print('diffusers:', diffusers.__version__)


In [ ]:
# ② 上传语音包（voice-pack.zip，在弹出的窗口里选文件）
import os, zipfile, json
from google.colab import files

uploaded = files.upload()  # 选 voice-pack.zip
zip_name = next(iter(uploaded))
os.makedirs('voice_pack', exist_ok=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall('voice_pack')
print('解压完成：', sorted(os.listdir('voice_pack')))
pack = json.load(open('voice_pack/segments.json', encoding='utf-8'))
ref_text = open('voice_pack/ref_text.txt', encoding='utf-8').read().strip()
print('声音：', pack.get('voice_name'), '| 段数：', len(pack['segments']), '| 原文已填：', bool(ref_text))

In [ ]:
# ③ 加载模型（Qwen3-TTS-12Hz-1.7B-Base，克隆专用）
# T4 是老架构：用 float16 + sdpa（不要 flash-attn，T4 上容易装不上）
import torch
from qwen_tts import Qwen3TTSModel

model = Qwen3TTSModel.from_pretrained(
    'Qwen/Qwen3-TTS-12Hz-1.7B-Base',
    device_map='cuda:0',
    dtype=torch.float16,
    attn_implementation='sdpa',
)
print('模型加载完成')

In [ ]:
# ⑥ 一键启动（T4）：Drive 语音包 → 解包 → 加载模型 → 起 Gradio
# 跑完 ① 之后直接跑这格就行，不用再手动传语音包、不用跑 ②③④⑤。
# 语音包在 Google Drive：MyDrive/qwen3tts/voice_pack/test-voice-pack-taiyi.zip
# 跑完后 Gradio 公开链接会自动登记到注册表，工作台打开就自动用这个 T4 地址，不用再手动贴链接。
# 注册表：https://voice-t4-registry.hench.workers.dev/t4-url（只存最新一条，12 小时过期）
# 保持这格运行中，停了工作台就连不上 T4。
import os, json, zipfile
import torch

DRIVE_ZIP = '/content/drive/MyDrive/qwen3tts/voice_pack/test-voice-pack-taiyi.zip'

# 1. 挂载 Google Drive（已挂载就跳过）
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive 已挂载')
else:
    print('Drive 已挂载，跳过')

# 2. 从 Drive 解包语音包
if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError(
        '语音包不在 Drive 上：' + DRIVE_ZIP +
        '。先把 test-voice-pack-taiyi.zip 存到 Google Drive 的 qwen3tts/voice_pack/ 文件夹里。')
os.makedirs('voice_pack', exist_ok=True)
with zipfile.ZipFile(DRIVE_ZIP) as z:
    z.extractall('voice_pack')
print('解压完成：', sorted(os.listdir('voice_pack')))
pack = json.load(open('voice_pack/segments.json', encoding='utf-8'))
ref_text = open('voice_pack/ref_text.txt', encoding='utf-8').read().strip()
print('声音：', pack.get('voice_name'), '| 段数：', len(pack['segments']), '| 原文已填：', bool(ref_text))

# 3. 加载模型（前面跑过 ③ 就不再重复加载）
try:
    model
    print('模型已加载，跳过。')
except NameError:
    from qwen_tts import Qwen3TTSModel
    model = Qwen3TTSModel.from_pretrained(
        'Qwen/Qwen3-TTS-12Hz-1.7B-Base',
        device_map='cuda:0',
        dtype=torch.float16,
        attn_implementation='sdpa',
    )
    print('模型加载完成')

# 4. 起 Gradio：接口与 HF Space 的 /generate_voice_clone 同款参数，
#    工作台填了这里的公开链接就能直接调 T4 生成
import gradio as gr

REF_WAV = 'voice_pack/ref.wav'

def generate_voice_clone(ref_audio, ref_text, target_text, language, use_xvector_only, model_size):
    # model_size 只是兼容 HF Space 的参数名：T4 这台只加载了 1.7B，就用 1.7B 生成
    kwargs = dict(text=target_text, language=language or 'Chinese', ref_audio=ref_audio)
    if use_xvector_only:
        kwargs['x_vector_only_mode'] = True
        kwargs['ref_text'] = None
    else:
        kwargs['ref_text'] = ref_text or '你好'
    wavs, sr = model.generate_voice_clone(**kwargs)
    return (sr, wavs[0]), 'T4 生成完成（1.7B）'

demo = gr.Interface(
    fn=generate_voice_clone,
    inputs=[
        gr.Audio(type='filepath', value=REF_WAV, label='Reference Audio'),
        gr.Textbox(value=ref_text, label='Reference Text'),
        gr.Textbox(label='Target Text'),
        gr.Dropdown(choices=['Chinese', 'English'], value='Chinese', label='Language'),
        gr.Checkbox(value=False, label='Use x-vector only'),
        gr.Dropdown(choices=['1.7B', '0.6B'], value='1.7B', label='Model Size'),
    ],
    outputs=[
        gr.Audio(label='Generated Audio'),
        gr.Textbox(label='Status'),
    ],
    api_name='generate_voice_clone',
    title='Qwen3-TTS 声音克隆（Colab T4）',
)
# 5. 把 Gradio 公开链接登记到注册表（工作台打开就自动读，不用手动贴）
import time
REGISTRY_URL = 'https://voice-t4-registry.hench.workers.dev/t4-url'
print('正在启动 Gradio（share=True）……')
demo.launch(share=True, prevent_thread_lock=True)
public_url = None
for _ in range(30):
    public_url = getattr(demo, 'share_url', None)
    if public_url:
        break
    time.sleep(2)
print('Gradio 公开链接：', public_url)
if public_url:
    try:
        import urllib.request
        req = urllib.request.Request(REGISTRY_URL, data=json.dumps({'url': public_url}).encode('utf-8'), headers={'Content-Type': 'application/json'}, method='POST')
        with urllib.request.urlopen(req, timeout=15) as resp:
            print('已登记到工作台（打开工作台会自动用这个 T4 地址）：', resp.status)
    except Exception as e:
        print('登记失败（不影响生成，手动把链接贴到工作台就行）：', e)
else:
    print('没拿到 Gradio 公开链接，手动复制上面 Running on public URL 的链接贴到工作台。')
# 挂起保持服务在线（不要停掉这格，停了工作台就连不上 T4）
print('服务已启动，工作台会自动读取这个地址。保持这格运行中，关闭即断开。')
while True:
    time.sleep(3600)


In [ ]:
# ④ 逐段生成（每段约几十秒）
import time, soundfile as sf

os.makedirs('output', exist_ok=True)
for seg in pack['segments']:
    i, text = seg['index'], seg['text']
    t0 = time.time()
    kwargs = dict(text=text, language='Chinese', ref_audio='voice_pack/ref.wav')
    if ref_text:
        kwargs['ref_text'] = ref_text
    else:
        # 没填原文：只用说话人向量克隆（效果稍差，但不用原文）
        kwargs['x_vector_only_mode'] = True
        kwargs['ref_text'] = None
    wavs, sr = model.generate_voice_clone(**kwargs)
    out = f'output/voice_{i:02d}.wav'
    sf.write(out, wavs[0], sr)
    print(f'第 {i}/{len(pack["segments"])} 段完成：{out}（{time.time()-t0:.0f} 秒）')
print('全部生成完成')

In [ ]:
# ⑤ 打包下载
import shutil
shutil.make_archive('voice-output', 'zip', 'output')
files.download('voice-output.zip')
print('已下载 voice-output.zip，每段一个 wav')